<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/tf_idf_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# Base directory (where tfidf_input_*.tsv lives)
BASE_DIR = "/content/drive/MyDrive/world-inflation/result/tsv"

# Choose target subreddit: "food", "cars", "real_estate", "travel", "frugal"
SUBREDDIT = "frugal"

# N-gram range for TF-IDF: (2, 2) = bigrams only, (1, 2) = uni + bi
NGRAM_RANGE = (2, 2)

# Minimum document frequency (absolute count)
MIN_DF_DOCS = 5    # tune if needed

# Maximum document frequency (fraction of documents, 1.0 disables it)
MAX_DF_FRAC = 0.95

# Top-K terms to display for positive/negative TF-IDF differences
TOP_K = 20

# NEW: minimum total document frequency across BEFORE+AFTER
MIN_DOCFREQ_TOTAL = 10  # e.g., 10 or 15; can be tuned


In [ ]:
# Period settings per subreddit: (start_year_month, end_year_month)
# These can be edited for sensitivity analysis.
PERIOD_CONFIG = {
    "food": {
        "before": ("2018-04", "2021-05"),
        "after":  ("2021-06", "2022-12"),
    },
    "cars": {
        "before": ("2018-07", "2020-09"),
        "after":  ("2020-10", "2021-12"),
    },
    "real_estate": {
        "before": ("2016-05", "2021-02"),
        "after":  ("2021-03", "2022-07"),
    },
    "travel": {
        "before": ("2013-08", "2020-06"),
        "after":  ("2020-07", "2022-12"),
    },
    "frugal": {
        "before": ("2015-02", "2021-03"),
        "after":  ("2021-04", "2022-01"),
    },
}

# Scraping keywords to exclude per subreddit (collection-time filters)
EXCLUDE_KEYWORDS = {
    "food": [
        "food", "like", "price", "cost", "inflation", "deflation",
        "expensive", "cheap", "purchase", "sale"
    ],
    "cars": [
        "like", "price", "cost", "inflation", "deflation",
        "expensive", "cheap", "purchase", "sale"
    ],
    "real_estate": [
        "real", "estate", "like", "price", "cost", "inflation",
        "deflation", "expensive", "cheap", "purchase", "sale"
    ],
    "travel": [
        "travel", "like", "price", "cost", "inflation", "deflation",
        "expensive", "cheap", "purchase", "sale",
        'america', 'united states', 'usa', 'the us', 'u.s.', 'stateside', 'across the states',
        'alabama', 'alaska', 'arizona', 'arkansas', 'california', 'colorado', 'connecticut', 'delaware', 'florida',
        'georgia', 'hawaii', 'idaho', 'illinois', 'indiana', 'iowa', 'kansas', 'kentucky', 'louisiana', 'maine',
        'maryland', 'massachusetts', 'michigan', 'minnesota', 'mississippi', 'missouri', 'montana', 'nebraska',
        'nevada', 'new hampshire', 'new jersey', 'new mexico', 'new york', 'north carolina', 'north dakota', 'ohio',
        'oklahoma', 'oregon', 'pennsylvania', 'rhode island', 'south carolina', 'south dakota', 'tennessee', 'texas',
        'utah', 'vermont', 'virginia', 'washington', ' DC ', 'west virginia', 'wisconsin', 'wyoming',
        'new york', 'ny', 'nyc', 'los angeles', 'chicago', 'houston', 'phoenix', 'philadelphia', 'san antonio', 'san diego',
        'dallas', 'san jose', 'austin', 'jacksonville', 'fort worth', 'columbus', 'indianapolis', 'charlotte', 'san francisco', 'seattle',
        'nashville', 'denver', 'oklahoma city', 'el paso', 'boston', 'portland', 'las vegas', 'vegas', 'detroit', 'memphis', 'louisville',
        'baltimore', 'milwaukee', 'albuquerque', 'tucson', 'fresno', 'sacramento', 'kansas city', 'mesa', 'atlanta', 'omaha',
        'colorado springs', 'raleigh', 'long beach', 'virginia beach', 'miami', 'oakland', 'minneapolis', 'tulsa', 'bakersfield',
        'wichita', 'arlington', 'aurora', 'tampa', 'new orleans', 'cleveland', 'honolulu', 'anaheim', 'lexington', 'stockton',
        'corpus christi', 'henderson', 'riverside', 'newark', 'st. paul', 'santa ana', 'cincinnati', 'irvine', 'orlando', 'pittsburgh',
        'st. louis', 'greensboro', 'jersey city', 'anchorage', 'lincoln', 'plano', 'durham', 'buffalo', 'chandler', 'chula vista',
        'toledo', 'madison', 'gilbert', 'reno', 'fort wayne', 'north las vegas', 'st. petersburg', 'lubbock', 'irving', 'laredo',
        'winston-salem', 'chesapeake', 'glendale', 'garland', 'scottsdale', 'norfolk', 'boise', 'fremont', 'spokane', 'santa clarita',
        'baton rouge', 'richmond', 'hialeah',
        'grand canyon', 'yellowstone', 'hollywood', 'niagara', 'disney world', 'yosemite', 'central park',
        'amtrak', 'greyhound', 'interstate', 'york', 'san', 'los'
    ],
    "frugal": [
        "frugal", "like", "price", "cost", "inflation", "deflation",
        "expensive", "cheap", "purchase", "sale"
    ],
}

# Construct stopword list: English stopwords + scraping keywords
subreddit_excludes = EXCLUDE_KEYWORDS.get(SUBREDDIT, [])
STOPWORD_SET = set(ENGLISH_STOP_WORDS).union(set(subreddit_excludes))
STOPWORDS = list(STOPWORD_SET)

print(f"Subreddit: {SUBREDDIT}")
print("Period config:", PERIOD_CONFIG[SUBREDDIT])
print("Number of stopwords:", len(STOPWORDS))


In [ ]:
def remove_urls(text: str) -> str:
    """Remove URLs and typical URL tokens from text."""
    if not isinstance(text, str):
        return ""
    # Remove URL patterns
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Remove residual url-like tokens
    text = re.sub(r"\bhttps?\b", " ", text)
    text = re.sub(r"\bwww\b", " ", text)
    text = re.sub(r"\bcom\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load_and_split_corpus(subreddit: str) -> tuple:
    """
    Load tfidf_input_<subreddit>.tsv, ensure year_month exists,
    filter inflation==2, split into before/after corpora based on PERIOD_CONFIG.
    Returns: before_docs (list[str]), after_docs (list[str])
    """
    input_path = os.path.join(BASE_DIR, f"rerun-tfidf_input_{subreddit}.tsv")
    print("Loading:", input_path)
    df = pd.read_csv(input_path, sep="\t")

    # Ensure inflation is numeric and filter inflation==2 (inflation only)
    if "inflation" not in df.columns:
        raise ValueError("Column 'inflation' not found in input file.")
    df["inflation"] = pd.to_numeric(df["inflation"], errors="coerce")
    df = df[df["inflation"] == 2].copy()

    # Ensure we have body_clean; fall back to body if necessary
    if "body_clean" in df.columns:
        df["text"] = df["body_clean"].astype(str)
    elif "body" in df.columns:
        df["text"] = df["body"].astype(str)
    else:
        raise ValueError("Neither 'body_clean' nor 'body' found in input file.")

    # URL removal as a final safety step
    df["text"] = df["text"].apply(remove_urls)

    # Ensure year_month exists; if not, derive from created_date
    if "year_month" not in df.columns:
        if "created_date" in df.columns:
            dt = pd.to_datetime(df["created_date"], errors="coerce")
            df["year_month"] = dt.dt.strftime("%Y-%m")
        else:
            raise ValueError("Need either 'year_month' or 'created_date' column.")

    # Filter by period config
    period = PERIOD_CONFIG[subreddit]
    b_start, b_end = period["before"]
    a_start, a_end = period["after"]

    before_mask = (df["year_month"] >= b_start) & (df["year_month"] <= b_end)
    after_mask  = (df["year_month"] >= a_start) & (df["year_month"] <= a_end)

    df_before = df[before_mask].copy()
    df_after  = df[after_mask].copy()

    print(f"Total inflation==2 rows: {len(df)}")
    print(f"Before period [{b_start}, {b_end}]: {len(df_before)} docs")
    print(f"After  period [{a_start}, {a_end}]: {len(df_after)} docs")

    # Drop empty texts
    df_before = df_before[df_before["text"].str.strip().astype(bool)]
    df_after  = df_after[df_after["text"].str.strip().astype(bool)]

    before_docs = df_before["text"].tolist()
    after_docs  = df_after["text"].tolist()

    print(f"Non-empty before docs: {len(before_docs)}")
    print(f"Non-empty after docs:  {len(after_docs)}")

    if len(before_docs) == 0 or len(after_docs) == 0:
        raise ValueError("One of the periods has zero documents. Check date ranges.")

    return before_docs, after_docs


In [ ]:
before_docs, after_docs = load_and_split_corpus(SUBREDDIT)

# Quick sanity check: print first examples
print("\nExample BEFORE doc:")
print(before_docs[0][:300], "...")
print("\nExample AFTER doc:")
print(after_docs[0][:300], "...")


In [ ]:
def compute_tfidf_diff(before_docs, after_docs):
    """
    Compute TF-IDF for before/after corpora using a shared vocabulary,
    then return a DataFrame with mean scores and their differences.
    """
    # Combine docs for a joint vocabulary
    all_docs = before_docs + after_docs

    vectorizer = TfidfVectorizer(
        stop_words=STOPWORDS,
        ngram_range=NGRAM_RANGE,
        min_df=MIN_DF_DOCS,
        max_df=MAX_DF_FRAC
    )

    X_all = vectorizer.fit_transform(all_docs)  # shape: (n_before + n_after, V)
    n_before = len(before_docs)
    n_after = len(after_docs)

    X_before = X_all[:n_before, :]
    X_after  = X_all[n_before:, :]

    # Mean TF-IDF per term in each period
    before_mean = np.asarray(X_before.mean(axis=0)).ravel()
    after_mean  = np.asarray(X_after.mean(axis=0)).ravel()

    # Document frequency (total / by period)
    df_total  = np.asarray((X_all > 0).sum(axis=0)).ravel()
    df_before = np.asarray((X_before > 0).sum(axis=0)).ravel()
    df_after  = np.asarray((X_after > 0).sum(axis=0)).ravel()

    terms = vectorizer.get_feature_names_out()

    df_scores = pd.DataFrame({
        "term": terms,
        "tfidf_before_mean": before_mean,
        "tfidf_after_mean": after_mean,
        "delta_after_minus_before": after_mean - before_mean,
        "docfreq_total": df_total,
        "docfreq_before": df_before,
        "docfreq_after": df_after,
    })

    # NEW: filter out very rare terms across both periods
    df_scores = df_scores[df_scores["docfreq_total"] >= MIN_DOCFREQ_TOTAL].copy()

    # Sort by absolute delta for inspection if needed
    df_scores["abs_delta"] = df_scores["delta_after_minus_before"].abs()  # CHANGED (after filter)

    return df_scores.sort_values("abs_delta", ascending=False).reset_index(drop=True)


tfidf_diff_df = compute_tfidf_diff(before_docs, after_docs)
print("TF-IDF diff table shape:", tfidf_diff_df.shape)
tfidf_diff_df.head()


# foods

In [ ]:
def extract_top_k(df_scores, top_k=20):
    """
    Extract top-K terms where TF-IDF increased (after > before)
    and where it decreased (before > after).
    """
    # After > Before
    pos_df = df_scores[df_scores["delta_after_minus_before"] > 0].copy()
    pos_df = pos_df.sort_values("delta_after_minus_before", ascending=False)
    pos_top = pos_df.head(top_k)

    # Before > After
    neg_df = df_scores[df_scores["delta_after_minus_before"] < 0].copy()
    neg_df = neg_df.sort_values("delta_after_minus_before", ascending=True)
    neg_top = neg_df.head(top_k)

    return pos_top, neg_top


pos_top, neg_top = extract_top_k(tfidf_diff_df, TOP_K)

print("\n=== Top terms (AFTER > BEFORE) ===")
print(pos_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

print("\n=== Top terms (BEFORE > AFTER) ===")
print(neg_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

# Save full diff table for later use (e.g., LaTeX table, appendix)
output_path = os.path.join(BASE_DIR, f"tfidf_diff_{SUBREDDIT}.csv")
tfidf_diff_df.to_csv(output_path, index=False)
print(f"\nFull TF-IDF diff table saved to: {output_path}")


# cars

In [ ]:
def extract_top_k(df_scores, top_k=20):
    """
    Extract top-K terms where TF-IDF increased (after > before)
    and where it decreased (before > after).
    """
    # After > Before
    pos_df = df_scores[df_scores["delta_after_minus_before"] > 0].copy()
    pos_df = pos_df.sort_values("delta_after_minus_before", ascending=False)
    pos_top = pos_df.head(top_k)

    # Before > After
    neg_df = df_scores[df_scores["delta_after_minus_before"] < 0].copy()
    neg_df = neg_df.sort_values("delta_after_minus_before", ascending=True)
    neg_top = neg_df.head(top_k)

    return pos_top, neg_top


pos_top, neg_top = extract_top_k(tfidf_diff_df, TOP_K)

print("\n=== Top terms (AFTER > BEFORE) ===")
print(pos_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

print("\n=== Top terms (BEFORE > AFTER) ===")
print(neg_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

# Save full diff table for later use (e.g., LaTeX table, appendix)
output_path = os.path.join(BASE_DIR, f"tfidf_diff_{SUBREDDIT}.csv")
tfidf_diff_df.to_csv(output_path, index=False)
print(f"\nFull TF-IDF diff table saved to: {output_path}")


# real_estate

In [ ]:
def extract_top_k(df_scores, top_k=20):
    """
    Extract top-K terms where TF-IDF increased (after > before)
    and where it decreased (before > after).
    """
    # After > Before
    pos_df = df_scores[df_scores["delta_after_minus_before"] > 0].copy()
    pos_df = pos_df.sort_values("delta_after_minus_before", ascending=False)
    pos_top = pos_df.head(top_k)

    # Before > After
    neg_df = df_scores[df_scores["delta_after_minus_before"] < 0].copy()
    neg_df = neg_df.sort_values("delta_after_minus_before", ascending=True)
    neg_top = neg_df.head(top_k)

    return pos_top, neg_top


pos_top, neg_top = extract_top_k(tfidf_diff_df, TOP_K)

print("\n=== Top terms (AFTER > BEFORE) ===")
print(pos_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

print("\n=== Top terms (BEFORE > AFTER) ===")
print(neg_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

# Save full diff table for later use (e.g., LaTeX table, appendix)
output_path = os.path.join(BASE_DIR, f"tfidf_diff_{SUBREDDIT}.csv")
tfidf_diff_df.to_csv(output_path, index=False)
print(f"\nFull TF-IDF diff table saved to: {output_path}")


# travel

In [ ]:
def extract_top_k(df_scores, top_k=20):
    """
    Extract top-K terms where TF-IDF increased (after > before)
    and where it decreased (before > after).
    """
    # After > Before
    pos_df = df_scores[df_scores["delta_after_minus_before"] > 0].copy()
    pos_df = pos_df.sort_values("delta_after_minus_before", ascending=False)
    pos_top = pos_df.head(top_k)

    # Before > After
    neg_df = df_scores[df_scores["delta_after_minus_before"] < 0].copy()
    neg_df = neg_df.sort_values("delta_after_minus_before", ascending=True)
    neg_top = neg_df.head(top_k)

    return pos_top, neg_top


pos_top, neg_top = extract_top_k(tfidf_diff_df, TOP_K)

print("\n=== Top terms (AFTER > BEFORE) ===")
print(pos_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

print("\n=== Top terms (BEFORE > AFTER) ===")
print(neg_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

# Save full diff table for later use (e.g., LaTeX table, appendix)
output_path = os.path.join(BASE_DIR, f"tfidf_diff_{SUBREDDIT}.csv")
tfidf_diff_df.to_csv(output_path, index=False)
print(f"\nFull TF-IDF diff table saved to: {output_path}")


# frugal

In [ ]:
def extract_top_k(df_scores, top_k=20):
    """
    Extract top-K terms where TF-IDF increased (after > before)
    and where it decreased (before > after).
    """
    # After > Before
    pos_df = df_scores[df_scores["delta_after_minus_before"] > 0].copy()
    pos_df = pos_df.sort_values("delta_after_minus_before", ascending=False)
    pos_top = pos_df.head(top_k)

    # Before > After
    neg_df = df_scores[df_scores["delta_after_minus_before"] < 0].copy()
    neg_df = neg_df.sort_values("delta_after_minus_before", ascending=True)
    neg_top = neg_df.head(top_k)

    return pos_top, neg_top


pos_top, neg_top = extract_top_k(tfidf_diff_df, TOP_K)

print("\n=== Top terms (AFTER > BEFORE) ===")
print(pos_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

print("\n=== Top terms (BEFORE > AFTER) ===")
print(neg_top[["term", "tfidf_before_mean", "tfidf_after_mean", "delta_after_minus_before",
               "docfreq_before", "docfreq_after"]])

# Save full diff table for later use (e.g., LaTeX table, appendix)
output_path = os.path.join(BASE_DIR, f"tfidf_diff_{SUBREDDIT}.csv")
tfidf_diff_df.to_csv(output_path, index=False)
print(f"\nFull TF-IDF diff table saved to: {output_path}")
